In [5]:
# pandas for loading and working with the CSV as a dataframe
import pandas as pd

# Task 2 gets its own notebook, so reload the raw data fresh —
# nothing carries over from 01_profile.ipynb's kernel.
df = pd.read_csv("../data/raw_pos_export.csv")

# Sanity check: should be (131, 17), matching Task 1's profiling.
df.shape

(131, 17)

In [6]:
# Fix: pandas inferred one format from the first rows and coerced everything
# else to NaT instead of trying each row's format individually.
# format="mixed" tells it to parse each row on its own terms.
df["ts_parsed"] = pd.to_datetime(df["transaction_ts"], format="mixed", errors="coerce")

# Should now be close to 1 (the single blank transaction_ts from Task 1),
# not 108 like the naive parse produced.
df["ts_parsed"].isna().sum()

np.int64(1)

In [7]:
# Drop rows where parsing still failed (or was genuinely blank) so they don't
# poison the sort comparison the way the earlier 108 NaTs did.
valid = df.dropna(subset=["ts_parsed"]).copy()

# argsort() gives the row order each column would need to be sorted ascending.
# If row_id and ts_parsed produce the same order, row_id is a safe proxy
# for chronological order.
row_id_order = valid["row_id"].argsort()
ts_order = valid["ts_parsed"].argsort()

# Count of positions where the two orderings disagree.
# 0 = row_id is a perfectly reliable time-ordering proxy.
# Any non-zero count = row_id can drift from true chronological order,
# which matters for how tightly you can trust "nearby rows = nearby time"
# when you build the invoice reconstruction rule in Task 3.
(row_id_order.values != ts_order.values).sum()

np.int64(124)

In [ ]:
df

,row_id,store_id,register_id,cashier_name,transaction_ts,invoice_number,customer_ref,customer_name_raw,customer_email_raw,product_sku,product_name_raw,category_raw,qty,unit_price_raw,discount_code,payment_method,line_note,ts_parsed
0,1,ST01,REG-3,tom reyes,2025-03-03 09:10:04,NaN,C002,James Chen,jchen@example.com,SKU-104,ceramic mug,Home,-1,$15.36,WELCOME10,Cash,RETURN,2025-03-03 09:10:04
1,2,ST01,REG-3,tom reyes,2025-03-03 09:10:03,NaN,C002,James Chen,jchen@example.com,SKU-101,Blue T-Shirt (M),APPAREL,3,18.28,NaN,Cash,NaN,2025-03-03 09:10:03
2,3,ST01,REG-3,tom reyes,2025-03-03 09:10:11,NaN,C002,James Chen,jchen@example.com,SKU-106,Enamel Pin Set,Accessories,2,$9.61,WELCOME10,Cash,NaN,2025-03-03 09:10:11
3,4,ST01,REG-3,tom reyes,2025-03-03 09:10:27,NaN,C002,James Chen,jchen@example.com,SKU-110,Reusable Water Bottle,Home,1,$34.07,VOLUNTEER15,Mastercard,NaN,2025-03-03 09:10:27
4,5,ST99,REG-2,T. Reyes,03/03/2025 09:20 AM,INV-1000,C003,Priya Nair,priya.nair@example.com,SKU-102,Blue T-Shirt (L),Apparel,2,27.76,STAFF20,AMEX,NaN,2025-03-03 09:20:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,127,ST02,REG-3,Maria Gomez,04-Mar-2025 06:10,NaN,C002,James Chen,jchen@example.com,SKU-102,Blue T-Shirt (L),Apparel,-1,19.61,NaN,Debit,RETURN,2025-03-04 06:10:00
127,128,ST02,REG-3,Maria Gomez,04-Mar-2025 06:09,NaN,C002,James Chen,jchen@example.com,SKU-109,Sticker Pack,Stationery,2,$40.29,STAFF20,Debit,NaN,2025-03-04 06:09:00
128,129,ST02,REG-3,Maria Gomez,04-Mar-2025 06:09,NaN,C002,James Chen,jchen@example.com,SKU-109,Sticker Pack,Stationery,2,14.41,STAFF20,Cash,NaN,2025-03-04 06:09:00
129,130,ST02,REG-3,Maria Gomez,04-Mar-2025 06:10,NaN,C002,James Chen,jchen@example.com,SKU-110,Reusable Water Bottle,Home,2,17.12,NaN,Mastercard,NaN,2025-03-04 06:10:00
